In [1]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import Lasso
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.decomposition import IncrementalPCA

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
df = pd.read_csv("final-Project.csv", on_bad_lines='skip')

/tmp/ipykernel_9070/3245767582.py:1: DtypeWarning: Columns (19) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("final-Project.csv", on_bad_lines='skip')


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 448175 entries, 0 to 448174
Data columns (total 21 columns):
 #   Column                                                       Non-Null Count   Dtype 
---  ------                                                       --------------   ----- 
 0   Numéro d'identification / Number ID                          448175 non-null  int64 
 1   Date Received / Date reçue                                   448175 non-null  object
 2   Complaint Received Type                                      448175 non-null  object
 3   Type de plainte reçue                                        448175 non-null  object
 4   Country                                                      448175 non-null  object
 5   Pays                                                         448174 non-null  object
 6   Province/State                                               448172 non-null  object
 7   Province/État                                                448172 non-nu

In [ ]:
df.isnull().sum()

In [ ]:
df.dropna(inplace=True)
df.info()

In [ ]:
df['Number of Victims / Nombre de victimes'] = df['Number of Victims / Nombre de victimes'].fillna(0).astype(int)
df['Dollar Loss /pertes financières'] = df['Dollar Loss /pertes financières'].replace({r'[$,]': ''}, regex=True).astype(float)
df

In [ ]:
df.info()

In [ ]:
df.columns

In [ ]:
df = df.rename(columns={
    "Numéro d'identification / Number ID": "Number ID",
    "Date Received / Date reçue": "Date Received",
    "Victim Age Range / Tranche d'âge des victimes" : "Victim Age Range",
    "Number of Victims / Nombre de victimes":"Number of Victims",
    "Dollar Loss /pertes financières": "Dollar Loss"
})
display(df.head())

# should i drop pays? if not please let me know what should i do ?

In [ ]:
columns_to_drop = [
    'Pays',
    'Genre',
    'Type de plainte',
    'Province/État',
    'Catégories thématiques sur la fraude et la cybercriminalité',
    'Langue de correspondance',
    'Méthode de sollicitation',
    'Type de plainte reçue'
]

existing_columns_to_drop = [col for col in columns_to_drop if col in df.columns]
df = df.drop(columns=existing_columns_to_drop)

display(df.columns)

In [ ]:
df['Victim Age Range'] = df['Victim Age Range'].str.lstrip("' ")

In [ ]:
df

# work on date and keep it in your data set!

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# df = df.drop(columns=['_id', 'Number ID', 'Date Received'])

# print(df.shape)
# df.head()

In [ ]:


# ===== 1. Dollar Loss Distribution =====
plt.figure(figsize=(10,4))
df['Dollar Loss'].dropna().hist(bins=50)
plt.title('Distribution of Dollar Loss')
plt.xlabel('Dollar Loss')
plt.show()

# ===== 2. Top Provinces =====
df['Province/State'].value_counts().head(10).plot(kind='bar', figsize=(10,4), color='steelblue')
plt.title('Cases by Province/State')
plt.tight_layout()
plt.show()

# ===== 3. Victim Age Range =====
df['Victim Age Range'].value_counts().plot(kind='bar', figsize=(10,4), color='coral')
plt.title('Victim Age Range Distribution')
plt.tight_layout()
plt.show()

# ===== 4. Solicitation Method =====
if 'Solicitation Method' in df.columns:
    df['Solicitation Method'].value_counts().plot(kind='bar', figsize=(10,4), color='mediumseagreen')
    plt.title('Solicitation Method')
    plt.tight_layout()
    plt.show()

# ===== 5. Complaint Type =====
if 'Complaint Type' in df.columns:
    df['Complaint Type'].value_counts().head(10).plot(kind='bar', figsize=(12,4), color='mediumpurple')
    plt.title('Top 10 Complaint Types')
    plt.tight_layout()
    plt.show()

# ===== 6. Dollar Loss by Age Range (Boxplot) =====
plt.figure(figsize=(12,5))
df.dropna(subset=['Victim Age Range', 'Dollar Loss']).boxplot(
    column='Dollar Loss', by='Victim Age Range', figsize=(12,5)
)
plt.title('Dollar Loss by Age Range')
plt.suptitle('')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


# Scatter diagram

In [ ]:
# #draw histogram of each feature
# df.hist(bins=50, figsize=(20,15))
# #save_fig("attribute_histogram_plots")
# plt.show()

In [ ]:
# ===== 7. Full Correlation Heatmap
df_encoded = df.copy()
cat_cols = df_encoded.select_dtypes(include='object').columns

for col in cat_cols:
    df_encoded[col] = pd.factorize(df_encoded[col])[0]

# just numeric columns
numeric_df = df_encoded.select_dtypes(include='number')

plt.figure(figsize=(12,8))
sns.heatmap(numeric_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap - All Features')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
sns.scatterplot(data=df, x='Number of Victims', y='Dollar Loss', hue='Victim Age Range')
plt.title('Number of Victims vs Dollar Loss')
plt.tight_layout()
plt.show()

In [ ]:
print("data set beore encoding:", df.shape)

In [ ]:
df_encoded = df.copy()
df_encoded = df_encoded.drop(columns=['Number ID', 'Date Received'])
print("encoded data set:", df_encoded.shape)

In [ ]:
df_encoded = df.copy()
df_encoded = df_encoded.drop(columns=['Number ID', 'Date Received'])

# ===== 1. Ordinal Encoding =====
# Victim Age Range
age_order = ['Under 18', '18 - 19', '20 - 29', '30 - 39',
             '40 - 49', '50 - 59', '60 - 69', '70 - 79',
             '80 and over', 'Not Available / non disponible']
df_encoded['Victim Age Range'] = df_encoded['Victim Age Range'].map(
    {age: i for i, age in enumerate(age_order)}
).fillna(-1).astype(int)

# Better version (recommended) for One-Hot Encoding
categorical_cols = [
    'Country',
    'Province/State',
    'Fraud and Cybercrime Thematic Categories',
    'Solicitation Method',
    'Gender',
    'Language of Correspondence',
    'Complaint Received Type',
    'Complaint Type'
]

df_encoded = pd.get_dummies(df_encoded,
                            columns=categorical_cols,
                            drop_first=True,
                            dtype='int8')   # saves memory

print("After One-Hot Encoding → Shape:", df_encoded.shape)
print("Dtypes:\n", df_encoded.dtypes.value_counts())
df_encoded.head()

## Model Selection and Evaluation

In this project, the objective is to predict financial loss using the **“Dollar Loss”** variable as the target. Since the target variable is continuous, the problem is formulated as a **regression task**.

As a baseline, a **Linear Regression** model is implemented to establish a reference level of performance. This model assumes a linear relationship between the input features and the target variable.

To improve performance and reduce overfitting, two regularized models are also applied:

- **Ridge Regression (L2 regularization)**, which penalizes large coefficients and improves model stability.  
- **Lasso Regression (L1 regularization)**, which performs feature selection by shrinking some coefficients to zero.  

In addition, the **“Dollar Loss”** variable is highly skewed. Therefore, a **log transformation** is applied to normalize the distribution and enhance model performance.

All models are trained on the scaled feature set, and evaluated on the validation dataset using the following metrics:

- Mean Absolute Error (MAE)  
- Mean Squared Error (MSE)  
- Root Mean Squared Error (RMSE)  

These metrics are used to compare model performance and determine the most suitable approach for predicting financial loss.

In [ ]:

# log transform for dollar loss cuz we have skewed in this feature.
df_encoded['Dollar Loss Log'] = np.log1p(df_encoded['Dollar Loss'])

# X و y
X = df_encoded.drop(columns=['Dollar Loss', 'Dollar Loss Log'])
y = df_encoded['Dollar Loss Log']

from sklearn.model_selection import train_test_split

# first split: train + temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# second split: validation + test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)
print("Train:", X_train.shape)
print("Val:", X_val.shape)
print("Test:", X_test.shape)


# scaled data

In [ ]:

scaler = MinMaxScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

In [ ]:
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

y_pred = lr.predict(X_val_scaled)

mae = mean_absolute_error(y_val, y_pred)
mse = mean_squared_error(y_val, y_pred)
rmse = mse ** 0.5

print("Linear Regression MAE:", mae)
print("Linear Regression MSE:", mse)
print("Linear Regression RMSE:", rmse)

In [ ]:


lasso = Lasso()

param_grid = {
    'alpha': [0.001, 0.01, 0.1, 1]
}

grid_lasso = GridSearchCV(
    lasso,
    param_grid,
    cv=5,
    scoring='neg_mean_absolute_error'
)

grid_lasso.fit(X_train_scaled, y_train)

best_lasso = grid_lasso.best_estimator_
y_pred_val = best_lasso.predict(X_val_scaled)

print("Best alpha:", grid_lasso.best_params_)
print("Lasso MAE:", mean_absolute_error(y_val, y_pred_val))
print("Lasso MSE:", mean_squared_error(y_val, y_pred_val))
print("Lasso RMSE:", mean_squared_error(y_val, y_pred_val) ** 0.5)


In [ ]:

ridge = Ridge()

param_grid = {
    'alpha': [0.01, 0.1, 1, 10]
}

grid_ridge = GridSearchCV(
    ridge,
    param_grid,
    cv=5,
    scoring='neg_mean_absolute_error'
)

grid_ridge.fit(X_train_scaled, y_train)

best_ridge = grid_ridge.best_estimator_
y_pred_val = best_ridge.predict(X_val_scaled)

mae = mean_absolute_error(y_val, y_pred_val)
mse = mean_squared_error(y_val, y_pred_val)
rmse = mse ** 0.5

print("Best alpha:", grid_ridge.best_params_)
print("Ridge MAE:", mae)
print("Ridge MSE:", mse)
print("Ridge RMSE:", rmse)

In [ ]:
# from sklearn.preprocessing import PolynomialFeatures
# from sklearn.linear_model import Ridge
# from sklearn.pipeline import Pipeline
# from sklearn.model_selection import GridSearchCV
# from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ridge_pipe = Pipeline([
#     ('poly', PolynomialFeatures(degree=2, include_bias=False, interaction_only=True)),
#     ('model', Ridge())
# ])

# param_grid = {
#     'poly__degree': [2],
#     'model__alpha': [0.1, 1, 10]
# }

# grid_ridge_poly = GridSearchCV(
#     ridge_pipe,
#     param_grid,
#     cv=5,
#     n_jobs=1,
#     verbose=1
# )

# grid_ridge_poly.fit(X_train_scaled, y_train)

# print(" Best parameters:", grid_ridge_poly.best_params_)

# # Evaluate
# y_pred_poly = grid_ridge_poly.predict(X_val_scaled)

# print("\n=== Polynomial Regression (degree=2) + Ridge ===")
# print(f"MAE  : {mean_absolute_error(y_val, y_pred_poly):.4f}")
# print(f"RMSE : {mean_squared_error(y_val, y_pred_poly)**0.5:.4f}")
# print(f"R²   : {r2_score(y_val, y_pred_poly):.4f}")

In [ ]:
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    n_jobs=-1,
    random_state=42
)

rf.fit(X_train, y_train)        # No scaling needed for Random Forest
y_pred_rf = rf.predict(X_val)

print("=== Random Forest Regressor ===")
print(f"MAE  : {mean_absolute_error(y_val, y_pred_rf):.4f}")
print(f"RMSE : {mean_squared_error(y_val, y_pred_rf)**0.5:.4f}")
print(f"R²   : {r2_score(y_val, y_pred_rf):.4f}")

In [ ]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
  n_estimators=300,
  max_depth=6,
  learning_rate=0.1,
  subsample=0.8,
  colsample_bytree=0.8,
  n_jobs=-1,
  random_state=42
)
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_val)

print(f"XGBoost MAE : {mean_absolute_error(y_val, y_pred_xgb):.4f}")
print(f"XGBoost R² : {r2_score(y_val, y_pred_xgb):.4f}")

In [ ]:
models = {
    "Linear Regression": lr,
    "Ridge": best_ridge,
    "Lasso": best_lasso,
    "Random Forest": rf,
    "XGBoost": xgb,

}

print("=== FINAL COMPARISON: Validation vs Test Set ===\n")
print(f"{'Model':<25} {'Val MAE':<10} {'Test MAE':<10} {'Val RMSE':<10} {'Test RMSE':<10} {'Val R²':<8} {'Test R²':<8}")
print("-" * 85)

for name, model in models.items():
    # Validation prediction
    if name == "Random Forest":
        y_pred_val = model.predict(X_val)
    else:
        y_pred_val = model.predict(X_val_scaled)

    # Test prediction
    if name == "Random Forest":
        y_pred_test = model.predict(X_test)
    else:
        y_pred_test = model.predict(X_test_scaled)

    # Metrics
    mae_val = mean_absolute_error(y_val, y_pred_val)
    mae_test = mean_absolute_error(y_test, y_pred_test)
    rmse_val = mean_squared_error(y_val, y_pred_val)**0.5
    rmse_test = mean_squared_error(y_test, y_pred_test)**0.5
    r2_val = r2_score(y_val, y_pred_val)
    r2_test = r2_score(y_test, y_pred_test)

    print(f"{name:<25} {mae_val:.4f}    {mae_test:.4f}    {rmse_val:.4f}    {rmse_test:.4f}    {r2_val:.4f}   {r2_test:.4f}")

    # Quick overfitting check
    diff = mae_test - mae_val
    if diff > 0.1:
        print(f"   → Possible overfitting (Test worse by {diff:.4f})")
    elif diff < -0.05:
        print(f"   → Possible underfitting or lucky split")
    print()

In this project, we aimed to predict the financial loss (`Dollar Loss`) from Canadian fraud and cybercrime complaints using a regression approach.

After applying proper preprocessing steps — including:
- log transformation (`np.log1p`) to handle the highly skewed target variable,
- one-hot encoding for categorical features,
- and feature scaling,

we trained and compared four models: **Linear Regression**, **Ridge**, **Lasso**, and **Random Forest Regressor**.

**Final Results (Validation vs Test Set):**

- **Random Forest Regressor** achieved the best performance with:
  - Validation MAE = 0.8837
  - Test MAE = 0.8928
  - Test R² = 0.6932

The gap between validation and test performance is very small across all models, indicating **good generalization** and no significant overfitting.

Linear models (Linear Regression, Ridge, and Lasso) showed very similar results (MAE ≈ 1.45–1.46), while **Random Forest** clearly outperformed them by reducing MAE by more than 38%.

**Conclusion:**  
**Random Forest Regressor** is the most suitable model for predicting financial loss in this dataset. It effectively captures non-linear relationships and provides the highest predictive accuracy with excellent generalization on unseen data.

In [ ]:
# First pass: fit with None to get explained variance ratios for component selection
temp_pca = IncrementalPCA(n_components=None)
temp_pca.fit(X_train_scaled)

# Calculate the number of components needed to explain 95% variance
cumulative_variance_ratio = np.cumsum(temp_pca.explained_variance_ratio_)
# Find the first index where cumulative variance is >= 0.95
n_components_to_keep = np.where(cumulative_variance_ratio >= 0.95)[0][0] + 1 # +1 for 0-based index

# Second pass: initialize and fit IncrementalPCA with the determined number of components
pca = IncrementalPCA(n_components=n_components_to_keep)

# Fit on training data and transform validation + test sets
X_train_pca = pca.fit_transform(X_train_scaled)
X_val_pca   = pca.transform(X_val_scaled)
X_test_pca  = pca.transform(X_test_scaled)

# Results
print("PCA completed successfully!")
print(f"Number of components selected : {pca.n_components_}")
print(f"Total variance explained     : {pca.explained_variance_ratio_.sum():.3f} (95% target)")
print(f"Original shape               : {X_train_scaled.shape}")
print(f"New shape after PCA          : {X_train_pca.shape}")

In [ ]:
lr_pca = LinearRegression()
lr_pca.fit(X_train_pca, y_train)
y_pred_lr_pca_val = lr_pca.predict(X_val_pca)
y_pred_lr_pca_test = lr_pca.predict(X_test_pca)

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid_ridge = {'alpha': [0.01, 0.1, 1, 10, 100]}

grid_ridge_pca = GridSearchCV(
    Ridge(),
    param_grid_ridge,
    cv=5,
    scoring='neg_mean_absolute_error',
    n_jobs=-1
)
grid_ridge_pca.fit(X_train_pca, y_train)

best_ridge_pca = grid_ridge_pca.best_estimator_
y_pred_ridge_pca_val = best_ridge_pca.predict(X_val_pca)
y_pred_ridge_pca_test = best_ridge_pca.predict(X_test_pca)

print(f" Best Ridge alpha on PCA: {grid_ridge_pca.best_params_['alpha']}")


In [ ]:
param_grid_lasso = {'alpha': [0.001, 0.01, 0.1, 1, 10]}

grid_lasso_pca = GridSearchCV(
    Lasso(max_iter=5000),
    param_grid_lasso,
    cv=5,
    scoring='neg_mean_absolute_error',
    n_jobs=-1
)
grid_lasso_pca.fit(X_train_pca, y_train)

best_lasso_pca = grid_lasso_pca.best_estimator_
y_pred_lasso_pca_val = best_lasso_pca.predict(X_val_pca)
y_pred_lasso_pca_test = best_lasso_pca.predict(X_test_pca)

print(f" Best Lasso alpha on PCA: {grid_lasso_pca.best_params_['alpha']}")

In [ ]:

rf_pca = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    n_jobs=-1,
    random_state=42
)
rf_pca.fit(X_train_pca, y_train)
y_pred_rf_pca_val = rf_pca.predict(X_val_pca)
y_pred_rf_pca_test = rf_pca.predict(X_test_pca)

print("All models trained on PCA features with GridSearch!\n")

In [ ]:
print("FINAL COMPARISON - Original vs PCA (95% variance)\n")
print(f"{'Model':<20} {'Val MAE':<10} {'Test MAE':<10} {'Val RMSE':<10} {'Test RMSE':<10} {'Val R²':<8} {'Test R²':<8} {'Version':<10}")
print("-" * 85)

models_pca = {
    "Linear Regression": (y_pred_lr_pca_val, y_pred_lr_pca_test),
    "Ridge": (y_pred_ridge_pca_val, y_pred_ridge_pca_test),
    "Lasso": (y_pred_lasso_pca_val, y_pred_lasso_pca_test),
    "Random Forest": (y_pred_rf_pca_val, y_pred_rf_pca_test)
}

original_models = {
    "Linear Regression": lr,
    "Ridge": best_ridge,
    "Lasso": best_lasso,
    "Random Forest": rf
}

for name, (y_val_pred, y_test_pred) in models_pca.items():
    mae_val = mean_absolute_error(y_val, y_val_pred)
    mae_test = mean_absolute_error(y_test, y_test_pred)
    rmse_val = mean_squared_error(y_val, y_val_pred) ** 0.5
    rmse_test = mean_squared_error(y_test, y_test_pred) ** 0.5
    r2_val = r2_score(y_val, y_val_pred)
    r2_test = r2_score(y_test, y_test_pred)

    print(f"{name:<20} {mae_val:.4f}     {mae_test:.4f}     {rmse_val:.4f}     {rmse_test:.4f}     {r2_val:.4f}   {r2_test:.4f}   PCA")

## Conclusion

In this project, we aimed to predict the financial loss (**Dollar Loss**) from Canadian fraud and cybercrime complaints using a regression approach.

After proper preprocessing — including log transformation (`np.log1p`) to handle the highly skewed target, one-hot encoding for categorical features, and feature scaling — we trained and compared five models: Linear Regression, Ridge, Lasso, Random Forest Regressor, and XGBoost Regressor.

We evaluated two feature sets:
- **Original features** (299 features after encoding)
- **PCA-reduced features** (IncrementalPCA keeping 95% variance → 33 components)

### Final Results (Validation vs Test Set)

**Original Features (Best Performance):**
- **Random Forest Regressor** achieved the best results:
  - Validation MAE = 0.8837
  - Test MAE = 0.8928
  - Test R² = 0.6932
- XGBoost was very close (MAE ≈ 0.89–0.90) but slightly worse in MAE while showing higher R².

**PCA Mode (95% variance):**
- All models performed worse after PCA.
- Random Forest MAE increased to 0.9845 (Val) / 1.0090 (Test) — an ~11% drop in accuracy.
- Linear models (Ridge/Lasso) also showed higher error.

The small gap between validation and test performance across all models indicates **good generalization** and no major overfitting.

### Recommendation

**Random Forest Regressor on the original features** is the most suitable model for this dataset. It effectively captures non-linear relationships and delivers the lowest prediction error (MAE) with excellent generalization.

Although **Incremental PCA** successfully reduced dimensionality and improved training speed, it resulted in a noticeable loss of predictive power. This shows that dimensionality reduction is not always beneficial — especially when tree-based models are involved — as important non-linear signals can be lost.

**Final Conclusion:**  
**Random Forest Regressor (original features)** is the best-performing model for predicting financial loss in Canadian fraud complaints. It provides the highest accuracy and reliability for real-world use.